In [51]:
import pandas as pd
import numpy as np
from pyproj import Transformer
import time

start = time.time()

df = pd.read_csv("../data/df_adem_enedis_iris_69.csv")

In [52]:
transformer = Transformer.from_crs("EPSG:2154", "EPSG:4326", always_xy=True)
df["lon"], df["lat"] = transformer.transform(
    df["coordonnee_cartographique_x_ban"].values,
    df["coordonnee_cartographique_y_ban"].values
)

In [53]:
## Consommation & coût par m²
df["conso_m2"] = df["conso_5_usages_ep"] / df["surface_habitable_logement"]
df["cout_m2"] = df["cout_total_5_usages"] / df["surface_habitable_logement"]

## Ancienneté du logement
df["anciennete"] = 2025 - df["annee_construction"]

## Volume approximatif
df["volume_logement"] = df["surface_habitable_logement"] * df["hauteur_sous_plafond"]

In [54]:
def simplifie_zone(zone):
    if pd.isna(zone):
        return None
    zone = str(zone)
    if "H1" in zone:
        return "froid"
    elif "H2" in zone:
        return "tempéré"
    elif "H3" in zone:
        return "chaud"
    else:
        return "inconnu"

df["zone_clim_simplifiee"] = df["zone_climatique"].apply(simplifie_zone)

In [55]:
def classe_annee(annee):
    if pd.isna(annee):
        return None
    elif annee < 1948:
        return "avant_1948"
    elif annee < 1975:
        return "1949_1974"
    elif annee < 1990:
        return "1975_1989"
    elif annee < 2000:
        return "1990_1999"
    elif annee < 2012:
        return "2000_2011"
    else:
        return "apres_2012"

df["classe_annee_construction"] = df["annee_construction"].apply(classe_annee)

In [56]:
color_map = {
    "A": "#00FF00", "B": "#7FFF00", "C": "#FFFF00",
    "D": "#FFD700", "E": "#FFA500", "F": "#FF4500", "G": "#FF0000"
}
df["color_dpe"] = df["etiquette_dpe"].map(color_map)

df["etiquette_dpe"] = pd.Categorical(
    df["etiquette_dpe"],
    categories=["A", "B", "C", "D", "E", "F", "G"],
    ordered=True
)

In [57]:
print(len(df.columns))
print(len(df) * len(df.columns))

49
17556798


In [58]:
cols_to_drop = [
    "etiquette_ges", "methode_application_dpe", "coordonnee_cartographique_y_ban",
    "numero_dpe", "date_fin_validite_dpe", "inertie_lourde", "annee_construction",
    "version_dpe", "modele_dpe", "coordonnee_cartographique_x_ban",
    "date_reception_dpe", "codeiris", "cout_total_5_usages", "conso_5_usages_ep", "zone_clim_simplifiee", "type_installation_ecs", "code_departement_ban", "conso_moy_site_mwh",
    "type_logement_source", "code_region_ban", "conso_refroidissement_ep", "nombre_niveau_logement", "hauteur_sous_plafond" ,"type_installation_chauffage", "nombre_de_logements" 
]

df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

print(f"Colonnes supprimées : {len(cols_to_drop)}")
print(f"➡️ Nouveau nombre de colonnes : {df.shape[1]}\n")

print(len(df.columns))
print(len(df) * len(df.columns))

df.head()

Colonnes supprimées : 25
➡️ Nouveau nombre de colonnes : 24

24
8599248


,emission_ges_5_usages,nom_commune_ban,type_energie_principale_chauffage,qualite_isolation_murs,type_batiment,conso_ecs_ep,surface_habitable_logement,conso_chauffage_ep,isolation_toiture,etiquette_dpe,...,conso_totale_mwh,conso_moy_commune_mwh,lon,lat,conso_m2,cout_m2,anciennete,volume_logement,classe_annee_construction,color_dpe
0,547.5,ALBIGNY-SUR-SAÔNE,ÉLECTRICITÉ,INSUFFISANTE,APPARTEMENT,3481.2,39.9,12342.6,0.0,F,...,6429.986,4.559143,4.835338,45.863835,419.095238,31.027569,61.0,99.75,1949_1974,#FF4500
1,6286.9,ALBIGNY-SUR-SAÔNE,GAZ NATUREL,INSUFFISANTE,APPARTEMENT,2122.8,79.2,25430.3,0.0,F,...,6429.986,4.559143,4.835338,45.863835,362.345960,27.449495,51.0,198.00,1949_1974,#FF4500
2,1435.8,NEUVILLE-SUR-SAÔNE,GAZ NATUREL,INSUFFISANTE,APPARTEMENT,1956.8,48.2,4210.8,0.0,C,...,7800.521,3.265994,4.849158,45.880012,154.311203,15.699170,18.0,120.50,2000_2011,#FFFF00
3,2659.4,NEUVILLE-SUR-SAÔNE,GAZ NATUREL,BONNE,APPARTEMENT,2181.5,102.1,9343.3,0.0,C,...,7800.521,3.265994,4.849551,45.880679,127.791381,11.508325,78.0,285.88,avant_1948,#FFFF00
4,1044.3,ALBIGNY-SUR-SAÔNE,GAZ NATUREL,BONNE,APPARTEMENT,1753.7,43.1,2703.9,0.0,C,...,6429.986,4.559143,4.829662,45.871398,130.301624,14.895592,13.0,107.75,apres_2012,#FFFF00


In [59]:
output_path = "../data/df_adem_enedis_iris_69_prepared.parquet"
df.to_parquet(output_path, index=False)

print(f"Fichier préparé : {output_path}")
print(f"Nombre de lignes : {len(df):,}")
print(f"Temps total : {time.time() - start:.2f}s")

Fichier préparé : ../data/df_adem_enedis_iris_69_prepared.parquet
Nombre de lignes : 358,302
Temps total : 21.04s


In [60]:
df.to_csv("../data/df_adem_enedis_iris_69_prepared.csv.gz", index=False, compression="gzip")

In [61]:
df_sample = df.sample(n=1000, random_state=42) 
df_sample.to_csv("../data/df_adem_enedis_iris_69_sample.csv.gz", index=False, compression="gzip")
print(f"Fichier d'échantillon sauvegardé ({len(df_sample):,} lignes)")

Fichier d'échantillon sauvegardé (1,000 lignes)
